[**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/agentic-ai-systems/blob/main/Lessons/L01/05_semantic_search.ipynb)

# Semantic Search: Indexing and Retrieval

[Source: Build a semantic search engine with LangChain | LangChain](https://docs.langchain.com/oss/python/langchain/knowledge-base).

Here we will build a search engine over a PDF document. This will allow us to retrieve passages in the PDF that are similar to an input query.

This guide focuses on LangChain's abstractions related to indexing and retrieval of text data:

* [Documents and document loaders](https://docs.langchain.com/oss/python/integrations/document_loaders);
* [Text splitters](https://docs.langchain.com/oss/python/integrations/splitters);
* [Embeddings](https://docs.langchain.com/oss/python/integrations/text_embedding);
* [Vector stores](https://docs.langchain.com/oss/python/integrations/vectorstores) and [retrievers](https://docs.langchain.com/oss/python/integrations/retrievers).

1. **Indexing**
   1. **Input processing** – Transform raw data into structured documents
   2. **Embedding & storage** – Convert text into searchable vector representations
2. **Retrieval** – Find relevant information based on user queries

This *Relevant context* would later be augmented (added) into an LLM's context window, allowing the LLM to answer questions based on data. The term for this kind of interaction is called: **Retrieval Augmented Generation (RAG)**.


## Setup


### Installation

This tutorial requires the `langchain-community` and `pypdf` packages. Using [uv](https://docs.astral.sh/uv/):

```bash
%pip install langchain-community pypdf
```


For more details, see our [Installation guide](https://docs.langchain.com/oss/python/langchain/install).


## 1. Documents and document loaders

LangChain implements a [Document](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) abstraction, which is intended to represent a unit of text and associated metadata. It has three attributes:

* `page_content`: a string representing the content;
* `metadata`: a dict containing arbitrary metadata;
* `id`: (optional) a string identifier for the document.

The `metadata` attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) object often represents a chunk of a larger document.

We can generate sample documents when desired:

In [46]:
%pip install langchain-community pypdf

In [47]:
!pip install langchain-community langchain-huggingface faiss-cpu pypdf sentence-transformers nest_asyncio

In [48]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="AWS detalis course",
        metadata={"source": "/AWS Course - All Slides.pdf"},
    ),

]


However, the LangChain ecosystem implements [document loaders](https://docs.langchain.com/oss/python/langchain/retrieval#document-loaders) that [integrate with hundreds of common sources](https://docs.langchain.com/oss/python/integrations/document_loaders/). This makes it easy to incorporate data from these sources into your AI application.


In [49]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/AWS Course - All Slides.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

879


`PyPDFLoader` loads one [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) object per PDF page. For each, we can easily access:

* The string content of the page;
* Metadata containing the file name and page number.

In [50]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

AWS CERTIFIED SOLUTIONS ARCHITECT ASSOCIATE Student e-Notebook                                                      Version 1.0

{'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20210205072054Z00'00'", 'title': 'Course e-Notebook wout notes Color', 'author': 'Eissa Abousherif www.dolfined.com', 'moddate': "D:20210205072054Z00'00'", 'source': '/AWS Course - All Slides.pdf', 'total_pages': 879, 'page': 0, 'page_label': '1'}


### Splitting

For both information retrieval and downstream question-answering purposes, a page may be too coarse a representation. Our goal in the end will be to retrieve [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) objects that answer an input query, and further splitting our PDF will help ensure that the meanings of relevant portions of the document are not "washed out" by surrounding text.

We can use [text splitters](https://docs.langchain.com/oss/python/langchain/retrieval#text_splitters) for this purpose. Here we will use a simple text splitter that partitions based on characters. We will split our documents into chunks of 1000 characters with 200 characters of overlap between chunks. The overlap helps mitigate the possibility of separating a statement from important context related to it. We use the `RecursiveCharacterTextSplitter`, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases.

We set `add_start_index=True` so that the character index where each split Document starts within the initial Document is preserved as metadata attribute `start_index`.

In [51]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

930


## 2. Embeddings

Vector search is a common way to store and search over unstructured data (such as unstructured text). The idea is to store numeric vectors that are associated with the text. Given a query, we can [embed](https://docs.langchain.com/oss/python/langchain/retrieval#embedding_models) it as a vector of the same dimension and use vector similarity metrics (such as cosine similarity) to identify related text.

LangChain supports embeddings from [dozens of providers](https://docs.langchain.com/oss/python/integrations/text_embedding/). These models specify how text should be converted into a numeric vector. We use HuggingFace with the `all-mpnet-base-v2` sentence transformer:

```bash
%pip install langchain-huggingface sentence-transformers
```

In [52]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[-0.000768802419770509, -0.04732697457075119, -0.03251934051513672, -0.011239398270845413, 0.034204721450805664, -0.01058821752667427, 0.04397312179207802, -0.0033209279645234346, -0.012506824918091297, 0.00011961268319282681]


Armed with a model for generating text embeddings, we can next store them in a special data structure that supports efficient similarity search.


## 3. Vector stores

LangChain [VectorStore](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore) objects contain methods for adding text and [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) objects to the store, and querying them using various similarity metrics. They are often initialized with [embedding](https://docs.langchain.com/oss/python/langchain/retrieval#embedding_models) models, which determine how text data is translated to numeric vectors.

LangChain includes a suite of [integrations](https://docs.langchain.com/oss/python/integrations/vectorstores) with different vector store technologies. For this tutorial we use the **in-memory** vector store, which is lightweight and requires no extra infrastructure:

In [54]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

Having instantiated our vector store, we can now index the documents.

In [55]:
ids = vector_store.add_documents(documents=all_splits)

Note that most vector store implementations will allow you to connect to an existing vector store--  e.g., by providing a client, index name, or other information. See the documentation for a specific [integration](https://docs.langchain.com/oss/python/integrations/vectorstores) for more detail.

Once we've instantiated a [`VectorStore`](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore) that contains documents, we can query it. [VectorStore](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore) includes methods for querying:

* Synchronously and asynchronously;
* By string query and by vector;
* With and without returning similarity scores;
* By similarity and [maximum marginal relevance](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore.max_marginal_relevance_search) (to balance similarity with query to diversity in retrieved results).

The methods will generally include a list of [Document](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) objects in their outputs.

**Usage**

Embeddings typically represent text as a "dense" vector such that texts with similar meanings are geometrically close. This lets us retrieve relevant information just by passing in a question, without knowledge of any specific key-terms used in the document.

Return documents based on similarity to a string query:

In [56]:
results = vector_store.similarity_search(
    "What is the main topic of this document?"
)

print(results[0])

page_content='AWS Certified Solutions Architect Associate SAA-C02 Student e-Notebook               
                                                                             www.dolfined.com                                      
440 © DolfinED All rights reserved  
Amazon DocumentDB and Amazon Neptune' metadata={'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20210205072054Z00'00'", 'title': 'Course e-Notebook wout notes Color', 'author': 'Eissa Abousherif www.dolfined.com', 'moddate': "D:20210205072054Z00'00'", 'source': '/AWS Course - All Slides.pdf', 'total_pages': 879, 'page': 440, 'page_label': '441', 'start_index': 0}


Async query:

In [57]:
results = await vector_store.asimilarity_search("What is self-attention?")

print(results[0])

page_content='AWS Certified Solutions Architect Associate SAA-C02 Student e-Notebook               
                                                                             www.dolfined.com                                      
803 © DolfinED All rights reserved  
   
Amazon CognitoSecurity, Identity, and Access Management 
© DolfinEDAll rights reserved
AmazonCognitoprovidesauthentication,authorization,andusermanagementforwebandmobileapplications.ThemaintwocomponentsofCognitoare:•CognitoUserPools:Providesuserdirectoriesthatprovidesign-upandsign-inoptionsforusers.•CognitoIdentitypools:CanbeusedtograntAWScredentials(STS)toaccessAWSservices.Theycanbeusedseparatelyortogether.' metadata={'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20210205072054Z00'00'", 'title': 'Course e-Notebook wout notes Color', 'author': 'Eissa Abousherif www.dolfined.com', 'moddate': "D:20210205072054Z00'00'", 'source': '/AWS Course - All Slides.pdf'

Return scores:

In [58]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("How does the Transformer encoder work?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.39219678611199804

page_content='AWS Certified Solutions Architect Associate SAA-C02 Student e-Notebook               
                                                                             www.dolfined.com                                      
860 © DolfinED All rights reserved  
   
Amazon Elastic TranscoderAmazon Media, Mobile, End User, and IoT Services
© DolfinEDAll rights reserved
ElasticTranscodermanagesthecomplexityofrunningmediatranscodingjobsatascaleinAWS.•Itisusedtoconvertvideoandaudio(media)filesstoredinS3intosupportedoutputformatsforplaybackonuserdevices.•Itsupportswiderangeofoutputformats,resolutions,bitrates,andframerates.•Isapayforwhatyouuseservice(hasafreetier).' metadata={'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20210205072054Z00'00'", 'title': 'Course e-Notebook wout notes Color', 'author': 'Eissa Abousherif www.dolfined.com', 'moddate': "D:20210205072054Z00'00'", 'source': '/AWS Course

Return documents based on similarity to an embedded query:

In [59]:
embedding = embeddings.embed_query("What are the advantages of the Transformer over RNNs?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='843 © DolfinED All rights reserved  
   
Amazon Kinesis Data Stream vs. SQS -when to use what?AWS Analytics Services
© DolfinEDAll rights reserved
Kinesis Data StreamsSQSIntended useReal-time ingestion and processing of streaming big data.Reliable, highly scalable hosted queue for storing messages as they travel between computers.OrderingOrdering of records, and ability to read/replay records in the same order by several Kinesis applications.SQS FIFO queues can guarantee message ordering.Use when your requirements are any of the following:•Routing related records to the same record processor (consumer). •When we need ordering of records (Important in case we need to keep the order of logs messages the same at the consumer as they arrived from the producers). •When we need multiple applications to consume the records concurrently. •The ability to consume the same records few hours or couple of days later.' metadata={'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz 

Learn more:

* [API Reference](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore)
* [Integration-specific docs](https://docs.langchain.com/oss/python/integrations/vectorstores)


## 4. Retrievers

Vectorstores implement an `as_retriever` method that will generate a Retriever, specifically a [`VectorStoreRetriever`](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.base.VectorStoreRetriever.html). These retrievers include specific `search_type` and `search_kwargs` attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [60]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "What is the main topic of this document?",
        "What is self-attention?",
    ],
)

[[Document(id='cc9ab6d7-99b1-4a5e-ac6a-0cfa23038936', metadata={'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20210205072054Z00'00'", 'title': 'Course e-Notebook wout notes Color', 'author': 'Eissa Abousherif www.dolfined.com', 'moddate': "D:20210205072054Z00'00'", 'source': '/AWS Course - All Slides.pdf', 'total_pages': 879, 'page': 440, 'page_label': '441', 'start_index': 0}, page_content='AWS Certified Solutions Architect Associate SAA-C02 Student e-Notebook               \n                                                                             www.dolfined.com                                      \n440 © DolfinED All rights reserved  \nAmazon DocumentDB and Amazon Neptune')],
 [Document(id='d262d97d-3586-48ce-b6f8-94c6157febe1', metadata={'producer': 'macOS Version 10.15.7 (Build 19H114) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20210205072054Z00'00'", 'title': 'Course e-Notebook wout notes Color', '

`VectorStoreRetriever` supports search types of `"similarity"` (default), `"mmr"` (maximum marginal relevance, described above), and `"similarity_score_threshold"`. We can use the latter to threshold documents output by the retriever by similarity score.

## Key Takeaways

- **Indexing vs. retrieval** – Indexing turns raw data into structured documents and embeds them for search; retrieval finds relevant passages for a query. Together they form the basis of **RAG (Retrieval Augmented Generation)**.
- **Documents** – LangChain’s `Document` has `page_content`, `metadata`, and optional `id`. Use **document loaders** (e.g. `PyPDFLoader`) to load from PDFs and other sources.
- **Chunking** – Split documents into smaller chunks (e.g. with `RecursiveCharacterTextSplitter`) so retrieval returns focused passages; use **overlap** to avoid cutting off context.
- **Embeddings** – Text is turned into dense vectors (e.g. via HuggingFace `all-mpnet-base-v2`); similar meaning → similar vectors, enabling **semantic** (meaning-based) search instead of keyword match.
- **Vector stores** – Store and query embeddings (e.g. `InMemoryVectorStore`); support sync/async, by query or by vector, with or without scores, and **MMR** for diversity.
- **Retrievers** – Use `vector_store.as_retriever()` with `search_type` (`"similarity"`, `"mmr"`, `"similarity_score_threshold"`) and `search_kwargs` (e.g. `k`) for easy integration into chains and RAG pipelines.